# CP1 · Notebook 03 — Detección con Deep Learning

## Objetivo

Aplicar un **modelo de segmentación de carriles pre-entrenado** a las **mismas 14 imágenes** del notebook 02. No vas a entrenar nada: cargas un modelo ya entrenado, lo aplicas y mides tiempo de inferencia.

El objetivo no es "entender la arquitectura del modelo" sino **vivir el contraste** con el clásico:
- ¿Cuánto tarda? 
- ¿Qué imágenes acierta que el clásico fallaba?
- ¿Hay alguna imagen donde el clásico es mejor?

## Pipeline

```
Imagen RGB ──► resize 256×256 ──► /255 ──► ONNX inference ──► inverted mask (256×256)
                                                                       │
                                                                       ▼
  Overlay  ◄──  Polyfit por lado  ◄──  Cluster L/R por centroide  ◄──  invert + threshold
```

## Modelo

`cp1-lane-segmenter.onnx` (16.9 MB) — **U-Net depthwise** entrenado sobre **BDD100K** (licencia MIT, [nickpai/lane-detection-unet-ncnn](https://huggingface.co/nickpai/lane-detection-unet-ncnn)), exportado a ONNX limpio para este módulo.

**Contrato I/O**:
- Input: `image`, shape `(1, 3, 256, 256)` float32, RGB normalizado a `[0, 1]` (`/255`, **sin** mean/std de ImageNet).
- Output: `lane_mask_inverted`, shape `(1, 1, 256, 256)` float32 en `[0, 1]`.
- **Convención invertida**: valores **bajos** son **líneas de carril**, valores altos son fondo. Calculamos `lane_prob = 1 - output` para invertir.

Si no tienes el modelo, desde la raíz del CP:

```bash
python scripts/download_assets.py
```

**Tiempo estimado del notebook**: 8 min. La inferencia en CPU es ~60 ms/imagen.

**Requiere**: `01_setup.ipynb` ejecutado, `02_clasico_canny_hough.ipynb` ejecutado (necesitamos su `outputs/02_classical_summary.json` para comparar en el notebook 04).

## 0. Imports y verificación de dependencias

Si `onnxruntime` no está instalado, falló el `pip install -r requirements.txt`. Vuelve atrás.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
import json

try:
    import onnxruntime as ort
except ImportError:
    raise SystemExit(
        'onnxruntime no está instalado.\n'
        'Ejecuta: pip install onnxruntime==1.17.3'
    )

print(f'onnxruntime: {ort.__version__}')
print(f'providers disponibles: {ort.get_available_providers()}')

DATASET_DIR = Path('../datasets/lanes-subset')
MODELS_DIR  = Path('../models')
OUT_DIR     = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODELS_DIR / 'cp1-lane-segmenter.onnx'

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'No encuentro el modelo en {MODEL_PATH}.\n'
        f'Desde la raíz de CP1-carriles/ ejecuta:\n'
        f'    python scripts/download_assets.py\n'
        f'y vuelve a ejecutar esta celda.'
    )

print(f'\nModelo encontrado: {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)')

## 1. Cargar el modelo ONNX e inspeccionar I/O

Antes de usar un modelo es **buena práctica** inspeccionar sus inputs y outputs. Te dirá:
- Qué shape debe tener la imagen (resize y formato)
- Qué shape devuelve (cómo postprocesarlo)

ONNX Runtime con `CPUExecutionProvider` es lo que usaremos. No hay GPU.

In [ ]:
session = ort.InferenceSession(
    str(MODEL_PATH),
    providers=['CPUExecutionProvider'],
)

inputs  = session.get_inputs()
outputs = session.get_outputs()

print('Inputs:')
for inp in inputs:
    print(f'  {inp.name:25s}  shape={inp.shape}  dtype={inp.type}')
print('Outputs:')
for out in outputs:
    print(f'  {out.name:25s}  shape={out.shape}  dtype={out.type}')

# Contrato esperado:
#   input  'image'              (1, 3, 256, 256) tensor(float)
#   output 'lane_mask_inverted' (1, 1, 256, 256) tensor(float) en [0,1] (low = lane)
INPUT_NAME  = inputs[0].name
OUTPUT_NAME = outputs[0].name
INPUT_H, INPUT_W = 256, 256

## 2. Preprocesado

Cada CNN tiene su preprocesado canónico. Este modelo (UNet depthwise) usa:

1. Resize a `(W=256, H=256)` con `cv2.resize` (bilinear). Sí, **cuadrado** — el modelo se entrenó así sobre BDD100K.
2. BGR → RGB.
3. Normalizar a `[0, 1]` dividiendo entre 255. **No** ImageNet mean/std (el modelo no lo usa).
4. Reordenar a NCHW: `(1, 3, 256, 256)`.

**Por qué importa**: cambiar el preprocesado **rompe la red**. Si normalizas con mean/std de ImageNet, este modelo produce ruido. Es el bug nº1 al usar modelos pre-entrenados — siempre verifica la convención de su README.

In [ ]:
def preprocess(image_bgr):
    """BGR uint8 (H, W, 3) → tensor NCHW float32 (1, 3, 256, 256)."""
    orig_h, orig_w = image_bgr.shape[:2]
    resized = cv2.resize(image_bgr, (INPUT_W, INPUT_H), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = rgb.transpose(2, 0, 1)[None, ...].astype(np.float32)
    return tensor, (orig_h, orig_w)

## 3. Inferencia sobre una imagen — y medir tiempo en CPU

Mide 5 inferencias después de un *warmup* (la primera siempre es más lenta por allocaciones internas).

In [ ]:
easy_imgs = sorted((DATASET_DIR / 'easy').glob('*.png'))
img_bgr = cv2.imread(str(easy_imgs[0]))
print(f'Imagen test: {easy_imgs[0].name}  shape={img_bgr.shape}')

tensor, (orig_h, orig_w) = preprocess(img_bgr)
print(f'Tensor input: shape={tensor.shape}  dtype={tensor.dtype}')

# Warmup
_ = session.run([OUTPUT_NAME], {INPUT_NAME: tensor})

# 5 inferencias cronometradas
times = []
for _ in range(5):
    t0 = time.perf_counter()
    out = session.run([OUTPUT_NAME], {INPUT_NAME: tensor})[0]
    times.append((time.perf_counter() - t0) * 1000)

print(f'\nTiempo inferencia (1 imagen, CPU):')
print(f'  min={min(times):.0f} ms  media={np.mean(times):.0f} ms  max={max(times):.0f} ms')
print(f'Output shape: {out.shape}  dtype={out.dtype}')
print(f'Output min/max: {out.min():.3f} / {out.max():.3f}  (recuerda: low = lane)')

# Output shape esperada: (1, 1, 256, 256). Quitamos las dimensiones de batch+canal
raw_mask = out[0, 0]
print(f'raw_mask 2D shape: {raw_mask.shape}')

## 4. Postprocesado: invertir → mascara binaria → puntos de carril → curvas

El modelo devuelve un mapa denso 256×256 con valores donde **bajos = lane** y altos = fondo. Pasos:

1. **Invertir** la convención (`lane_prob = 1 - raw_mask`).
2. **Threshold** sobre `lane_prob` → mascara binaria.
3. Por cada fila de la mitad inferior, encontrar centroides de píxeles activados → puntos del carril.
4. **Clustering** simple izquierda/derecha por posición x.
5. **Polyfit** orden 2 por lado → curvas.
6. Reescalar al tamaño original y dibujar.

In [ ]:
LANE_THRESHOLD = 0.35  # threshold sobre lane_prob = (1 - raw_mask)

def to_lane_mask(raw_mask, threshold=LANE_THRESHOLD):
    lane_prob = 1.0 - raw_mask              # invertir convención
    return (lane_prob > threshold).astype(np.uint8)

mask = to_lane_mask(raw_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(1.0 - raw_mask, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('Lane probability (1 - raw)')
axes[1].axis('off')

axes[2].imshow(mask, cmap='gray')
axes[2].set_title(f'Lane mask binaria (thr={LANE_THRESHOLD})')
axes[2].axis('off')
plt.tight_layout()
plt.show()

print(f'Pixeles activos: {mask.sum()} de {mask.size}  ({100*mask.sum()/mask.size:.2f}%)')

### 4.1 Extracción de puntos de carril por fila

Por cada fila de la mitad inferior del mask, encontramos clusters horizontales de píxeles activados y nos quedamos con su centroide. Cada centroide es un punto (x, y) de un carril.

In [ ]:
from scipy.ndimage import label as scipy_label


def extract_lane_points(mask, row_step=4, min_pixels=2):
    """Para cada fila (cada `row_step`) en la mitad inferior, devuelve los centroides
    de los runs horizontales de píxeles activados.
    
    Returns: lista de (x, y) en coordenadas del mask (256x256).
    """
    points = []
    h, w = mask.shape
    start_y = h // 2
    for y in range(start_y, h, row_step):
        row = mask[y]
        if row.sum() < min_pixels:
            continue
        labeled, n = scipy_label(row)
        for comp_id in range(1, n + 1):
            xs = np.where(labeled == comp_id)[0]
            if len(xs) < min_pixels:
                continue
            points.append((float(xs.mean()), float(y)))
    return points


def cluster_left_right(points, mask_w):
    mid_x = mask_w / 2.0
    left  = [(x, y) for x, y in points if x < mid_x]
    right = [(x, y) for x, y in points if x >= mid_x]
    return left, right


def fit_polynomial(points, order=2):
    """polyfit(y) → x (carriles son ~verticales)."""
    if len(points) < order + 1:
        return None
    xs = np.array([p[0] for p in points])
    ys = np.array([p[1] for p in points])
    return np.polyfit(ys, xs, deg=order)


points = extract_lane_points(mask)
left_pts, right_pts = cluster_left_right(points, mask.shape[1])
left_coeffs  = fit_polynomial(left_pts)
right_coeffs = fit_polynomial(right_pts)

print(f'Puntos detectados: {len(points)}  (L: {len(left_pts)}, R: {len(right_pts)})')
print(f'Polyfit L: {left_coeffs}')
print(f'Polyfit R: {right_coeffs}')

### 4.2 Reescalar y dibujar las curvas sobre la imagen original

In [ ]:
def draw_lanes_dl(image_bgr, left_coeffs, right_coeffs, mask_shape, color=(0, 200, 255), thickness=8):
    overlay = image_bgr.copy()
    H_in, W_in = mask_shape
    H_orig, W_orig = image_bgr.shape[:2]
    scale_x = W_orig / W_in
    scale_y = H_orig / H_in
    for coeffs in (left_coeffs, right_coeffs):
        if coeffs is None:
            continue
        ys_in = np.linspace(H_in // 2, H_in - 1, 50)
        xs_in = np.polyval(coeffs, ys_in)
        xs_orig = xs_in * scale_x
        ys_orig = ys_in * scale_y
        pts = np.stack([xs_orig, ys_orig], axis=1).astype(np.int32)
        cv2.polylines(overlay, [pts], isClosed=False, color=color, thickness=thickness)
    return cv2.addWeighted(image_bgr, 0.7, overlay, 0.3, 0)


result = draw_lanes_dl(img_bgr, left_coeffs, right_coeffs, mask.shape)
plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title('DL — overlay de carriles (curvas amarillas)')
plt.axis('off')
plt.show()

## 5. Empaquetar todo en una función

In [ ]:
def detect_lanes_dl(image_bgr, threshold=LANE_THRESHOLD):
    """Pipeline completo DL: imagen BGR → (overlay, left_coeffs, right_coeffs, time_ms)."""
    t0 = time.perf_counter()
    tensor, _ = preprocess(image_bgr)
    out = session.run([OUTPUT_NAME], {INPUT_NAME: tensor})[0]
    raw_mask = out[0, 0]
    mask = to_lane_mask(raw_mask, threshold)
    points = extract_lane_points(mask)
    left_pts, right_pts = cluster_left_right(points, mask.shape[1])
    left = fit_polynomial(left_pts)
    right = fit_polynomial(right_pts)
    overlay = draw_lanes_dl(image_bgr, left, right, mask.shape)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    return overlay, left, right, elapsed_ms

## 6. Aplicar a las 14 imágenes y medir tiempos

Pasamos las **mismas 14 imágenes** del notebook 02 por el modelo DL. En CPU son ~60 ms × 14 = <1 segundo de inferencia pura.

In [ ]:
from tqdm import tqdm

results = {'easy': [], 'medium': [], 'hard': []}
times_ms = []

for category in results.keys():
    folder = DATASET_DIR / category
    for img_path in tqdm(sorted(folder.glob('*.png')), desc=category):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        overlay, left, right, ms = detect_lanes_dl(img)
        results[category].append({
            'name': img_path.name,
            'overlay': overlay,
            'detected': (left is not None, right is not None),
            'time_ms': ms,
        })
        times_ms.append(ms)

print(f'\nProcesadas {sum(len(v) for v in results.values())} imágenes')
print(f'Tiempo medio:  {np.mean(times_ms):.0f} ms')
print(f'Tiempo p50:    {np.median(times_ms):.0f} ms')
print(f'Tiempo total:  {sum(times_ms) / 1000:.1f} s')

## 7. Grid visual por dificultad

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
for row, category in enumerate(['easy', 'medium', 'hard']):
    samples = results[category][:3]
    for col, sample in enumerate(samples):
        ax = axes[row, col]
        ax.imshow(cv2.cvtColor(sample['overlay'], cv2.COLOR_BGR2RGB))
        detected = 'L' if sample['detected'][0] else '-'
        detected += 'R' if sample['detected'][1] else '-'
        ax.set_title(f"{category} · {sample['name']} · [{detected}] · {sample['time_ms']:.0f}ms")
        ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Tasa de detección por categoría — y comparación con el clásico

In [ ]:
print(f"{'Categoría':<10} {'N':>4} {'Ambos':>8} {'Solo L/R':>10} {'Ninguno':>10}")
print('-' * 50)
for category, items in results.items():
    n = len(items)
    both = sum(1 for x in items if x['detected'] == (True, True))
    one  = sum(1 for x in items if sum(x['detected']) == 1)
    none_ = sum(1 for x in items if x['detected'] == (False, False))
    print(f'{category:<10} {n:>4} {both:>8} {one:>10} {none_:>10}')

classical_summary_path = OUT_DIR / '02_classical_summary.json'
if classical_summary_path.exists():
    with open(classical_summary_path) as f:
        classical = json.load(f)
    print('\nComparación tasa "ambos detectados":')
    print(f"{'cat':<10} {'clásico':>10} {'DL':>10} {'Δ':>8}")
    for cat in ['easy', 'medium', 'hard']:
        c_rate = classical['detection_rate'][cat]['both']
        d_rate = sum(1 for x in results[cat] if x['detected'] == (True, True)) / max(1, len(results[cat]))
        print(f'{cat:<10} {c_rate:>10.1%} {d_rate:>10.1%} {d_rate - c_rate:>+8.1%}')
else:
    print('\n(02_classical_summary.json no encontrado — ejecuta primero el notebook 02 para ver comparación)')

## 9. Distribución de tiempos de inferencia

El DL es más lento que el clásico, pero **¿cuánto?** Esto es relevante: a 30 FPS de cámara, el budget por frame es 33 ms. ¿Te da el modelo?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(times_ms, bins=10, color='#DA4544', edgecolor='black', alpha=0.8)
ax.axvline(np.mean(times_ms), color='black', linestyle='--', label=f'media={np.mean(times_ms):.0f} ms')
ax.axvline(33, color='green', linestyle=':', label='33 ms (30 FPS)')
ax.set_xlabel('Tiempo de inferencia (ms)')
ax.set_ylabel('Nº de imágenes')
ax.set_title('Distribución de tiempos DL en CPU (UNet depthwise, 256×256)')
ax.legend()
plt.tight_layout()
plt.show()

in_budget = np.mean(times_ms) <= 33
print(f'¿Llegamos a 30 FPS en CPU?  ',
      f"{'SÍ ✅' if in_budget else 'NO ❌ (este modelo es más pequeño que UFLDv2 pero todavía no entra en 33 ms con CPU general)'}")

## 10. Guardar resumen para el notebook 04 (comparativa)

In [ ]:
summary = {
    'method': 'dl_unet_depthwise_onnx',
    'model_file': MODEL_PATH.name,
    'mean_time_ms': float(np.mean(times_ms)),
    'median_time_ms': float(np.median(times_ms)),
    'p95_time_ms': float(np.percentile(times_ms, 95)),
    'total_images': sum(len(v) for v in results.values()),
    'detection_rate': {
        category: {
            'both': sum(1 for x in items if x['detected'] == (True, True)) / max(1, len(items)),
            'any':  sum(1 for x in items if any(x['detected'])) / max(1, len(items)),
        }
        for category, items in results.items()
    },
    'per_image': [
        {
            'name': x['name'],
            'category': category,
            'detected_left': bool(x['detected'][0]),
            'detected_right': bool(x['detected'][1]),
            'time_ms': float(x['time_ms']),
        }
        for category, items in results.items()
        for x in items
    ],
}

with open(OUT_DIR / '03_dl_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

overlays_dir = OUT_DIR / '03_dl_overlays'
overlays_dir.mkdir(exist_ok=True)
for category, items in results.items():
    for x in items:
        cv2.imwrite(str(overlays_dir / f"{category}_{x['name']}"), x['overlay'])

print(f'✅ Resumen guardado en {OUT_DIR / "03_dl_summary.json"}')
print(f'✅ {sum(len(v) for v in results.values())} overlays guardados en {overlays_dir}/')

## 11. Reflexión rápida (antes del notebook 04)

Mira **3 imágenes específicas**:

1. Una `easy` donde ambos métodos aciertan. ¿Cuál es "más limpio" visualmente? ¿Importa?
2. Una `hard` donde el clásico falló pero el DL acertó. ¿Qué hizo el DL? ¿Tiene sentido?
3. Una `hard` donde el DL también falla. ¿Qué cosa específica de la imagen lo despista?

**Apunta tus impresiones** — serán parte del entregable en `05_preguntas.md`.

## 12. Cierre

Has ejecutado un modelo DL en CPU. Mide y compara:

| | Clásico (Hough) | DL (ONNX UNet) |
|--|--|--|
| Tiempo medio | ~5–15 ms | ~60–100 ms |
| Setup | Cero | Modelo + onnxruntime |
| Robustez sombras | Mala | Mejor |
| Robustez curvas | Limitada | Mejor |
| Explicabilidad | Total | Limitada |

Ve a `04_comparativa.ipynb` para una comparativa cuantitativa y visual lado a lado.